#exploring data and analysing it

In [159]:

import numpy as np
import pandas as pd

df = pd.read_csv("../data/raw/metricses.csv")
df.head(5)

,timestamp,cpu,memory,latency,error_rate
0,2016-01-01 14:08:00,43.622600,74.541858,169.606237,1.850059
1,2016-01-01 08:26:00,44.076061,70.445812,347.454721,2.322975
2,2016-01-01 09:34:00,58.294056,66.996450,331.039655,0.729939
3,2016-01-01 02:19:00,37.691357,70.517591,126.623737,1.732359
4,2016-01-01 02:32:00,43.199753,50.415574,201.044324,0.000000


In [160]:
df.tail(5)

,timestamp,cpu,memory,latency,error_rate
995,2026-01-01 00:00:00,41.532063,68.250781,178.537779,2.892954
996,2016-01-01 04:16:00,62.669111,85.088586,244.344370,1.118932
997,2016-01-01 11:33:00,56.159356,39.810305,193.275161,3.057368
998,2016-01-01 02:25:00,57.818229,60.233686,274.493172,2.470044
999,2016-01-01 10:00:00,57.569886,57.256553,106.172365,1.176065


In [161]:
df.info()
print("*"*40)

print(df.shape)

print("*"*40)
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   timestamp   1000 non-null   str    
 1   cpu         951 non-null    float64
 2   memory      1000 non-null   float64
 3   latency     950 non-null    float64
 4   error_rate  1000 non-null   float64
dtypes: float64(4), str(1)
memory usage: 57.7 KB
****************************************
(1000, 5)
****************************************


,cpu,memory,latency,error_rate
count,951.000000,1000.000000,950.000000,1000.000000
mean,52.060400,61.062544,200.232872,4.487408
std,18.090876,14.961816,48.687838,20.364711
min,23.031134,15.894170,49.024392,0.000000
25%,43.482818,50.906375,167.977894,1.268018
50%,50.469806,60.946157,199.604459,2.029245
75%,56.840022,70.933233,232.371778,2.712420
max,197.525178,107.896614,396.311885,198.301073


In [162]:
df.isnull().sum()

timestamp      0
cpu           49
memory         0
latency       50
error_rate     0
dtype: int64

In [163]:
print(df["error_rate"].min())
print(df["error_rate"].max())

0.0
198.30107295651044


fix timestamps

In [164]:
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

df['timestamp'] = df['timestamp'].apply(
    lambda x: x.replace(year=2016) if pd.notna(x) and x.year == 2026 else x
)
df = df.sort_values('timestamp').reset_index(drop=True)

handling misssing values in data Interpolate() instead of fillna():
This is time-series data (metrics over time with a timestamp column).
interpolate() is better here because:

It fills missing values using neighboring points (linear by default) → preserves the trend and smoothness.
Much more realistic for CPU, memory, latency, etc.
Avoids sudden artificial jumps.

fillna() alternatives (and why they're worse here):

df['cpu'].fillna(df['cpu'].mean()) → fills with a constant → creates flat lines, distorts trends.
df['cpu'].fillna(method='ffill') or bfill → forward/backward fill → can create stair-step patterns.
These are okay for non-time-series data, but not ideal when you have ordered timestamps.

In [165]:
df['cpu'] = df['cpu'].interpolate(method='linear')
df['latency'] = df['latency'].interpolate(method='linear')

Cap outliers

In [167]:
for col in ['cpu', 'error_rate']:
    upper_limit = df[col].quantile(0.98)
    lower_limit = df[col].quantile(0.02)
    
    # Create cleaned versions
    df[f'{col}_clean'] = df[col].clip(lower=lower_limit, upper=upper_limit)
    
    # Flag outliers
    df[f'{col}_is_outlier'] = (df[col] < lower_limit) | (df[col] > upper_limit)

# Summary
print("New columns added:", [col for col in df.columns if 'clean' in col or 'outlier' in col])
print("\nOutlier counts:")
print(df[['cpu_is_outlier', 'error_rate_is_outlier']].sum())

print("\nFinal columns:", df.columns.tolist())
print(df.describe())



New columns added: ['cpu_clean', 'cpu_is_outlier', 'error_rate_clean', 'error_rate_is_outlier']

Outlier counts:
cpu_is_outlier           40
error_rate_is_outlier    20
dtype: int64

Final columns: ['timestamp', 'cpu', 'memory', 'latency', 'error_rate', 'cpu_clean', 'cpu_is_outlier', 'error_rate_clean', 'error_rate_is_outlier']
                        timestamp          cpu       memory      latency  \
count                        1000  1000.000000  1000.000000  1000.000000   
mean   2016-01-01 08:15:14.280000    51.972355    61.062544   200.406762   
min           2016-01-01 00:00:00    23.031134    15.894170    49.024392   
25%           2016-01-01 04:03:45    43.604151    50.906375   168.671309   
50%           2016-01-01 08:15:30    50.292505    60.946157   199.842385   
75%           2016-01-01 12:26:15    56.800735    70.933233   231.887741   
max           2016-01-01 16:39:00   164.988649   107.896614   396.311885   
std                           NaN    16.961152    14.961816   

In [168]:
df.isnull().sum()

timestamp                0
cpu                      0
memory                   0
latency                  0
error_rate               0
cpu_clean                0
cpu_is_outlier           0
error_rate_clean         0
error_rate_is_outlier    0
dtype: int64